Obtenemos el dataset (que ya viene separado en train, test y validation desde Tensorflow). Luego, se imprime la cantidad de elementos de cada conjunto.

In [ ]:
equiv = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes', '_silence_', '_unknown_']

In [ ]:
import tensorflow_datasets as tfds

(train, test, validation), info = tfds.load(
    "speech_commands",
    split=["train", "test", "validation"],
    with_info=True
)
print(len(train))
print(len(test))
print(len(validation))
print("=== Info general ===")
print(info)
print("=== Nombres de las etiquetas ===")
print(info.features['label'].names)
print("=== Cantidad de ejemplos por etiqueta en cada conjunto ===")
trainElements = [0] * 12
testElements = [0] * 12
valElements = [0] * 12

for e in train:
    trainElements[e['label'].numpy()] += 1

for e in test:
    testElements[e['label'].numpy()] += 1

for e in validation:
    valElements[e['label'].numpy()] += 1

print(trainElements)
print(testElements)
print(valElements)


In [ ]:
import IPython.display as ipd
for example in test:
    if (example['label'].numpy() == 11):
        audio = example['audio'].numpy()
        display(ipd.Audio(audio, rate=16000))
        break

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))
def getSpectrogram(audio):

    audio = tf.cast(audio, tf.float32) / 32768.0
    rightSamples = 16000
    audio = audio[:rightSamples]
    audio = tf.pad(audio, [[0, rightSamples - tf.shape(audio)[0]]])
    
    stft = tf.signal.stft(
        audio,
        frame_length=640,
        frame_step=320
    )

    spectrogram = tf.abs(stft)

    num_spectrogram_bins = spectrogram.shape[-1]
    num_mel_bins = 40
    sample_rate = 16000

    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins,
        num_spectrogram_bins,
        sample_rate
    )

    mel_spectrogram = tf.tensordot(
        spectrogram,
        mel_weight_matrix,
        1
    )

    mel_spectrogram.set_shape(
        spectrogram.shape[:-1].concatenate([num_mel_bins])
    )

    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)
    log_mel_spectrogram = log_mel_spectrogram[..., tf.newaxis]
    log_mel_spectrogram.set_shape((49, 40, 1))
    return log_mel_spectrogram

def showSpectrogram(spectrogram):
    spectrogram = tf.squeeze(spectrogram)
    plt.figure(figsize=(8,4))
    plt.imshow(tf.transpose(spectrogram), aspect='auto', origin='lower')
    plt.colorbar()
    plt.title("Spectrogram")
    plt.xlabel("Time")
    plt.ylabel("Frequency")
    plt.show()
i = 0
for audio in train:
    if (audio["label"].numpy() == 4):
        print(audio['audio'])
        showSpectrogram(getSpectrogram(audio['audio']))
        i += 1
        if i == 10:
            break
    

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input

def prepareData(e):
    audio = getSpectrogram(e["audio"])
    label = e["label"]
    return audio, label

unknown = train.filter(lambda x: x["label"] == 11)
known = train.filter(lambda x: x["label"] != 11)
unknown = unknown.shuffle(60000).take(6000)
count = unknown.reduce(0, lambda x, _: x + 1)
print("Unknown finales:", count.numpy())
train = known.concatenate(unknown)
train = train.shuffle(40000)

train1 = train.map(prepareData)
train1 = train1.batch(32)
validation1 = validation.map(prepareData).batch(32)
test1 = test.map(prepareData).batch(32)
for x, y in train1.take(1):
    print(x.shape, x.dtype)

numClasses = 12
def residual_block(x, filters):
    shortcut = x

    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1,1), padding="same")(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


inputs = Input(shape=(49,40,1))

x = layers.Conv2D(32, (3,3), padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

x = residual_block(x, 32)
x = layers.MaxPooling2D((2,2))(x)

x = residual_block(x, 64)
x = layers.MaxPooling2D((2,2))(x)

x = residual_block(x, 128)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(numClasses, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train1, validation_data=validation1, epochs=15)

loss, accuracy = model.evaluate(test1)

print(accuracy)

In [ ]:
loss, accuracy = model.evaluate(test1)

print(accuracy)
model.save("0.95_6000_dropout0.3_.keras")
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred_probs = model.predict(test1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.concatenate([y for x, y in test1], axis=0)

cm = confusion_matrix(y_true, y_pred)

print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.show()